# 113. 模型比较与基线

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 28 / 34 步：调优、比较、解释并保存模型**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** GridSearch与随机搜索  →  **本章任务：** 模型比较与基线  →  **下一步：** 特征选择与模型解释
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：拿到几个候选模型时，最有用的第一步不是直接挑分数最高的，而是先立一条最低可接受的基线，再把每个模型放回同一把尺子上。本章的表格和图从统一数据切分、统一指标开始，帮你把「模型相对基线的真实提升」和「得分波动」同时摊开来看，从而避免在微小差异上强行排名、把测试集当成排名训练资源这类常见误区。（打个比方：好比让几个候选人做同一套卷子、用同一评分标准，否则“分数高”可能只是因为题目更简单或批改更松。）




## 本章目标

学完本章，你将能够：

- **理解**：理解「模型比较与基线」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「模型比较与基线」的关键输出指标。
- **迁移**：能把「模型比较与基线」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 113.1 适用场景

拿到几个候选模型时，最有用的第一步不是直接挑分数最高的，而是先立一条最低可接受的基线，再把每个模型放回同一把尺子上。本章的表格和图从统一数据切分、统一指标开始，帮你把「模型相对基线的真实提升」和「得分波动」同时摊开来看，从而避免在微小差异上强行排名、把测试集当成排名训练资源这类常见误区。

**适合这样做的情形**：

- 需要判断复杂模型是否比均值或规则基线更值得投入；
- 需要在统一切分与统一指标下比较多个模型，并同时报告均值与波动；
- 需要确认一个看似更高的分数不是因为用了不同数据切分或不同指标换来的。


## 113.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | 参见本节示例 | 先明确样本、特征、目标和验证方式，再训练模型。 | 不同模型使用不同数据切分 |
| 模型、公式与诊断 | `models.items()`、`rows.append()`、`pd.DataFrame()`、`comparison.round()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 没有 Dummy 基线 |


## 113.3 示例 1：数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-113 -->
### 数学推导｜候选模型必须相对基线改进

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜同一验证折内做配对比较。** 第 $k$ 折差值 $d_k=s_{model,k}-s_{baseline,k}$。

**第 2 步｜平均差值而不只比较两个独立均值。** 

$$
\bar d=\frac1K\sum_kd_k,
\qquad
SD(d)=\sqrt{\frac1{K-1}\sum_k(d_k-\bar d)^2}
$$

**第 3 步｜再衡量相对提升。** 当基线不接近 0 时，可计算 $\bar d/|\bar s_{baseline}|$。分数提升还要与推理成本、延迟和维护复杂度一起评审。

**把上面的关系收束为本章计算式：**

$$
\Delta s=s_{model}-s_{baseline},\qquad relative\ gain=\frac{s_{model}-s_{baseline}}{|s_{baseline}|}
$$

**符号解释：** 同一切分、同一指标下的差值才具有可比性。

**代码对应：** 先建立 Dummy/简单规则基线，再比较候选模型和运行成本。

**使用边界：** 小幅分数提升可能不足以抵消复杂度、延迟和维护风险。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
cv = StratifiedKFold(5, shuffle=True, random_state=102)


**练一练**：数据与问题定义的手感练习

上一段示例里已经用 `X`、`y` 和 `cv` 把样本、特征与验证方式定义好，接下来体会「只改一个地方」如何影响模型结果。请复制 `X`，把其中某一列的整体数值放大（例如某一特征乘以 10），再用同一个 Logistic 模型、同一套 `cv` 评估一次，观察 `roc_auc` 均值的变化。运行前先写下你的预判，再把结果和预判对照。


In [ ]:
# 请在下方填写代码：只改一个数据字段，再评估基线 Logistic 的 AUC
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate


def roc_auc_mean(model, data, target):
    return cross_validate(model, data, target, cv=cv, scoring="roc_auc")[
        "test_score"
    ].mean()


logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

# TODO 1：基线 Logistic 在原始数据 X 上的 AUC
AUC_original = roc_auc_mean(logit, X, y)
print("原始数据 CV_AUC：", round(AUC_original, 4))

# TODO 2：复制 X，把某一列整体放大（例如乘 10），并记录你改了什么
X_changed = X.copy()
# 在此填写你要修改的列，例如：X_changed["某个特征"] = X_changed["某个特征"] * 10
changed_note = "待填写"

# TODO 3：用 X_changed 跑同样评估，得到 AUC_changed
AUC_changed = None  # 填空：AUC_changed = roc_auc_mean(logit, X_changed, y)


In [ ]:
# 完整答案：把 "mean radius" 列整体乘以 10，AUC 基本不变——缩放会被 StandardScaler 折抵
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate


def roc_auc_mean(model, data, target):
    return cross_validate(model, data, target, cv=cv, scoring="roc_auc")[
        "test_score"
    ].mean()


logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
AUC_original = roc_auc_mean(logit, X, y)

X_changed = X.copy()
X_changed["mean radius"] = X_changed["mean radius"] * 10  # 只改一个数据字段
AUC_changed = roc_auc_mean(logit, X_changed, y)

print("原始数据 CV_AUC：", round(AUC_original, 4))
print("修改后 CV_AUC：", round(AUC_changed, 4))
print("变化量：", round(AUC_changed - AUC_original, 4))


## 113.4 示例 2：模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd

models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logit": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "forest": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=102
    ),
}
rows = []
for name, m in models.items():
    s = cross_validate(
        m, X, y, cv=cv, scoring="roc_auc", return_train_score=True
    )
    rows.append(
        [
            name,
            s["test_score"].mean(),
            s["test_score"].std(),
            s["fit_time"].mean(),
        ]
    )
comparison = pd.DataFrame(
    rows, columns=["model", "CV_AUC", "std", "fit_seconds"]
).set_index("model")
display(comparison.round(4))


## 113.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 113.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 113.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 113.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 113.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 113.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 113.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 113.9 易错点提醒

- 不同模型使用不同数据切分
- 没有 Dummy 基线
- 分数差异极小时强行排名
- 忽略预测延迟、稳定性和可解释性


## 113.10 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 113.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 113.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 113.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
best_model = comparison.CV_AUC.idxmax()
improvement = (
    comparison.loc[best_model, "CV_AUC"] - comparison.loc["dummy", "CV_AUC"]
)
print(best_model, round(improvement, 3))


## 113.12 小结

用统一切分、统一指标和 Dummy 基线比较多个模型，同时报告性能与复杂度。


### 113.12.1 你已经掌握

- 建立分类基线
- 用相同 CV 比较模型
- 报告均值与波动
- 考虑训练时间和解释成本


### 113.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 113.12.3 需要注意

- 不同模型使用不同数据切分
- 没有 Dummy 基线
- 分数差异极小时强行排名
- 忽略预测延迟、稳定性和可解释性


### 113.12.4 完成检查

- [ ] 能够建立分类基线
- [ ] 能够用相同 CV 比较模型
- [ ] 能够报告均值与波动
- [ ] 能够考虑训练时间和解释成本


### 113.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
